In [ ]:
# analytics.query_data(start_time=None, end_time=None, username=None, board_name=None, db_path="kf.db", reverse=False)
# ↑ 按填入条件查询帖子，返回值有两个：①散装replies；②结构topics。
# ↑ ①所有符合条件的回复贴按回复时间顺序排序的列表，相较原始数据新增了reply_length字段（utf8字节数）
# ↑ ②主题帖新增reply_list/username/complete字段，reply_list中只含有该主题贴中所有符合条件的回复贴
# ↑ ②username用来指示该主题贴的发帖人，complete用来指示你是否解锁了该主题贴中所有的购买框和权限框

In [ ]:
import json, csv, os
from collections import Counter
from kf_analysis import analytics
import matplotlib.pyplot as plt
import numpy as np
from collections import Counter
# 设定图表中文字体，默认为更纱黑体Gothic
plt.rcParams["font.sans-serif"] = ["Sarasa Gothic SC"]
plt.rcParams["axes.unicode_minus"] = False
start_time, end_time = "2026-07-01 00:00:00", "2026-07-31 23:59:59"
replies, topics = analytics.query_data(start_time=start_time, end_time=end_time)

In [ ]:
# 简述A：统计期间，总活跃主题数，总新增回复数，总参与人数
# 简述B：天平均活跃主题数，天平均新增回复数，天平均参与人数
# 统计期间新增回复量热力图，天发言用户量折线图
# 统计期间各板块活跃主题量柱状图，统计期间各板块新增回复量柱状图
posters = len({r["username"] for r in replies})
days = (analytics.to_datetime(end_time).date() - analytics.to_datetime(start_time).date()).days + 1
print(f"统计期间：[{start_time[:-9]}, {end_time[:-9]}, {days}]\n"
      f"{posters} 账号参与了 {len(topics)} 主题的活跃，产生 {len(replies)} 回复。")
day_counts = Counter()
daily_topics = {}
daily_users = {}
for r in replies:
    d = analytics.to_datetime(r["reply_time"]).date()
    day_counts[d] += 1
    daily_topics.setdefault(d, set()).add(r["topic_id"])
    if r["username"]:
        daily_users.setdefault(d, set()).add(r["username"])
avg_topics = sum(len(s) for s in daily_topics.values()) / days
avg_users = sum(len(s) for s in daily_users.values()) / days
print(f"平均每天 {avg_topics:.1f} 主题活跃，{len(replies)/days:.1f} 回复新增，{avg_users:.1f} 账号发言。")
print("#1 本项目将主题第零楼也视作回复之一\n"
      "#2 主题统计口径不再是“统计期间新增的”，而是“统计期间活跃过的”\n"
      "#3 天均主题活跃量、天均发言账号量：(ΣTd)/D、(ΣUd)/D")
# calendar_heatmap函数可以通过cell_height参数调节格子高度
analytics.calendar_heatmap(day_counts, start_time, end_time, title="每天新增回复数量热力图", save="每天新增回复数量热力图")
boardlist = json.load(open("kf_analysis/configure.json", "r", encoding="utf-8"))["boardlist"]
board_order = {fid: i for i, (_, fid) in enumerate(boardlist)}
board_names = {t["board_id"]: t["board_name"] for t in topics}
topic_board_counts = Counter(t["board_id"] for t in topics)
reply_board_counts = Counter(r["board_id"] for r in replies)
boards = sorted(topic_board_counts, key=lambda fid: board_order.get(fid, len(board_order)))
labels = [board_names[fid] for fid in boards]
daily_dates = sorted(daily_users)
date_labels = [d.strftime("%m%d") for d in daily_dates]
analytics.output_plot_line(date_labels, [len(daily_users[d]) for d in daily_dates], title="每天发言用户数量", save="每天发言用户数量", color="skyblue", figsize=(16,3.5))
analytics.output_plot_bar(labels, [topic_board_counts[fid] for fid in boards], title="各板块活跃主题数量", save="各板块活跃主题数量", color="lightpink", figsize=(16,4.5))
analytics.output_plot_bar(labels, [reply_board_counts[fid] for fid in boards], title="各板块新增回复数量", save="各板块新增回复数量", color="mediumpurple", figsize=(16,4.5))

In [ ]:
# 用户活跃属性排行：回复数量 / 回复字节数 / 活跃天数比例
# exclude_boards用于在回复字节数统计时排除部分板块
exclude_boards = ["Galgame 网络硬盘区", "ACG音乐资源共享区", "CG画册资源共享区", "无限制资源区",
                  "动画资源共享区", "漫画轻小说共享区", "LIVE类资源分享区", "Galgame BitTorrent区", "GAL本子区"]
top_n = 100
reply_count = Counter()
reply_bytes = Counter()
active_days = {}
for r in replies:
    if not r["username"]: continue
    reply_count[r["username"]] += 1
    if r["board_name"] not in exclude_boards:
        reply_bytes[r["username"]] += r["reply_length"]
    active_days.setdefault(r["username"], set()).add(analytics.to_datetime(r["reply_time"]).date())
top_post = reply_count.most_common(top_n)
top_bytes = reply_bytes.most_common(top_n)
active_ratio = sorted(((n, len(d), len(d) / days * 100) for n, d in active_days.items()), key=lambda x: x[2], reverse=True)[:top_n]
def show(title, rows):
    print(f"\n{title} TOP{top_n}\n" + "="*30)
    for rank, (name, val) in enumerate(rows, 1):
        print(f"第{rank:>3}名 | {val} | {name}")
print("#1 回复字节数为回复内容的UTF8编码字节数。\n"
      "#2 回复字节数统计对象不包括九个资源区。")
show("回复数量", [(n, f"{c:>4} 回复") for n, c in top_post])
show("回复字节数", [(n, f"{c:>7} 字节") for n, c in top_bytes])
show("活跃天数比例", [(n, f"{d:>2}/{days} 天（{r:.2f}%）") for n, d, r in active_ratio])
with open("user_ranking.csv", "w", newline="", encoding="utf-8-sig") as f:
    writer = csv.writer(f)
    writer.writerow(["rank", "usernameA", "数量", "usernameB", "字节量", "usernameC", "天数", "比例%"])
    for i in range(top_n):
        a_name, a_val = top_post[i] if i < len(top_post) else ("", "")
        b_name, b_val = top_bytes[i] if i < len(top_bytes) else ("", "")
        c_name, c_days, c_ratio = active_ratio[i] if i < len(active_ratio) else ("", "", "")
        writer.writerow([i + 1, a_name, a_val, b_name, b_val, c_name, c_days,
                         round(c_ratio, 2) if c_ratio else ""])

In [ ]:
import re
top_n = 50
hb_color = "#FF0000"
normal_color = "#000000"
hb_title_pat = re.compile(r"hb", re.I)
hb_reply_pat = re.compile(r"(感谢|谢谢|多谢|有).{0,10}hb", re.I)
topic_day_counts = Counter()
for tids in daily_topics.values(): topic_day_counts.update(tids)
hot_stats = [(len(t["reply_list"]), topic_day_counts[t["topic_id"]], t) for t in topics]
def is_hb(t):
    return bool(hb_title_pat.search(t["topic_title"]) or any(hb_reply_pat.search(r["reply_text"] or "") for r in t["reply_list"]))
def show_hot(title, rows):
    print(f"\n{title} TOP{top_n}")
    for rank, (val, t) in enumerate(rows, 1):
        line = f"[b]no. {rank}: {val}: [/b][url={t['topic_url']}]{t['topic_title']}[/url]"
        print(f"[color={hb_color if is_hb(t) else normal_color}]{line}[/color]")
print("#1 通过标题与回复列表自动判断是否为hb相关贴，若是则以红色标注。")
show_hot("回复数量热帖榜", [(f"{c} replies", t) for c, days, t in sorted(hot_stats, key=lambda x: x[0], reverse=True)[:top_n]])
show_hot("讨论持续天数热帖榜", [(f"{days} days", t) for c, days, t in sorted(hot_stats, key=lambda x: x[1], reverse=True)[:top_n]])

In [ ]:
# 统计期间内，用户在九个资源区中的新增主题数量（总体与自购）
new_res_topics = [t for t in topics
                  if t["board_name"] in exclude_boards and t["username"]
                  and t["reply_list"] and t["reply_list"][0]["floor"] == 0]
post_counts = Counter(t["username"] for t in new_res_topics)
self_buy_counts = Counter(t["username"] for t in new_res_topics if "自购" in t["topic_title"])
def fmt_counts(counter):
    items = [f"{name} ({cnt})" for name, cnt in counter.most_common()]
    return "\n".join(", ".join(items[i:i + 5]) for i in range(0, len(items), 5))
print("总体数量：\n" + fmt_counts(post_counts) + "\n")
print("自购数量：\n" + fmt_counts(self_buy_counts) + "\n")
# 导出统计期间内，两个求助区所有新增主题的tid
help_tids = [f"https://bbs.kfpromax.com/read.php?tid={t["topic_id"]}&sf={t["topic_sf"]}" for t in topics
             if t["board_name"] in ("寻求资源", "图片/作品出处询问版")
             and t["reply_list"] and t["reply_list"][0]["floor"] == 0]
with open("求助区新增主题列表.txt", "w", encoding="utf-8") as f:
    f.write("\n".join(map(str, help_tids)))